In [31]:
import json
import re
import os
from pathlib import Path

Đọc file processed text

In [32]:
def read_processed_file(filepath):
    questions = []
    current_question = {}
    
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        
    for line in lines:
        line = line.strip()
        if line.startswith('Câu hỏi:'):
            if current_question:
                questions.append(current_question)
            current_question = {'question': line.replace('Câu hỏi:', '').strip()}
        elif line.startswith('Các đáp án:'):
            current_question['answers'] = []
        elif line.startswith('  ') and '. ' in line:
            current_question['answers'].append(line.split('. ')[1].strip())
        elif line.startswith('Đáp án đúng:'):
            current_question['correct_answer'] = line.replace('Đáp án đúng:', '').strip()
        elif line.startswith('Giải thích:'):
            current_question['explanation'] = line.replace('Giải thích:', '').strip()
    
    if current_question:
        questions.append(current_question)
    
    return questions

Tính toán các đặc trưng độ dài

In [33]:
def get_length_features(text):
    if not text:
        return {
            "char_length": 0,
            "word_length": 0,
            "token_length": 0
        }
    
    char_length = len(text)
    word_length = len(text.split())
    token_length = len(text.split())

    return {
        "char_length": char_length,
        "word_length": word_length,
        "token_length": token_length
    }


Lưu file

In [34]:
# Tạo thư mục features nếu chưa tồn tại
current_dir = Path.cwd()
base_dir = current_dir.parent
features_dir = current_dir / 'features'
features_dir.mkdir(exist_ok=True, parents=True)

# Xử lý từng môn học
subjects = ['văn', 'sử', 'địa', 'anh']

for subject in subjects:
    # Đọc file input
    input_file = base_dir / 'data_processed' / f'{subject}_processed.txt'
    output_file = features_dir / f'{subject}_length_features.txt'
    
    # Đọc và xử lý dữ liệu
    questions = read_processed_file(input_file)
    
    # Trích xuất features độ dài
    for q in questions:
        q['length_features'] = get_length_features(q['question'])
        q['answer_features'] = [get_length_features(ans) for ans in q['answers']]
    
    # Lưu kết quả
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(f"Features độ dài cho môn {subject}\n")
        f.write("-" * 50 + "\n\n")
        
        for i, q in enumerate(questions, 1):
            f.write(f"Câu {i}:\n")
            f.write(f"Câu hỏi: {q['question']}\n")
            f.write("Features câu hỏi:\n")
            for fname, fvalue in q['length_features'].items():
                f.write(f"  - {fname}: {fvalue}\n")
            f.write("Features câu trả lời:\n")
            for j, (ans, features) in enumerate(zip(q['answers'], q['answer_features']), 1):
                f.write(f"  {j}. {ans}\n")
                for fname, fvalue in features.items():
                    f.write(f"     - {fname}: {fvalue}\n")
            f.write("\n" + "-" * 50 + "\n\n")
    
    print(f"Đã lưu features độ dài cho môn {subject}")

Đã lưu features độ dài cho môn văn
Đã lưu features độ dài cho môn sử
Đã lưu features độ dài cho môn địa
Đã lưu features độ dài cho môn anh
